# 01 Baseline Inference: Zero-Shot vs Few-Shot

In [28]:
# setup and imports
import os
import sys
import random
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.model import DomainSummarizer, GenerationParams, FewShotExample
from src.evaluate import compute_rouge_batch, format_comparison_row, to_markdown_table

RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_FILE = RESULTS_DIR / "comparison_table.md"

print(f"Device available: {'cuda' if torch.cuda.is_available() else 'cpu'}")

Device available: cuda


In [29]:
# load a fixed 20-sample test subset and few-shot demonstrations
NUM_SAMPLES = 20

print("Loading test split...")
test_ds = load_dataset("cnn_dailymail", "3.0.0", split="test")
test_subset = test_ds.shuffle(seed=SEED).select(range(NUM_SAMPLES))

print("Loading tiny validation slice for few-shot demonstrations...")
val_demo = load_dataset("cnn_dailymail", "3.0.0", split="validation[:2]")
few_shot_examples = [
    FewShotExample(article=item["article"], summary=item["highlights"])
    for item in val_demo
]

articles = [item["article"] for item in test_subset]
references = [item["highlights"] for item in test_subset]

print(f"Prepared {len(articles)} test samples and {len(few_shot_examples)} few-shot examples.")

Loading test split...
Loading tiny validation slice for few-shot demonstrations...
Prepared 20 test samples and 2 few-shot examples.


In [30]:
# initialize summarizer and generation config
BASE_MODEL_NAME = "google/flan-t5-base"
generation_cfg = GenerationParams(
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    max_new_tokens=128,
)

summarizer = DomainSummarizer(model_name=BASE_MODEL_NAME, seed=SEED)

try:
    summarizer.load()
    print(f"Loaded model: {BASE_MODEL_NAME}")
except Exception as exc:
    raise RuntimeError(
        "Model loading failed. Check internet connectivity, available memory, "
        "and Hugging Face access."
    ) from exc

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded model: google/flan-t5-base


In [31]:
# zero-shot summarization
zero_shot_predictions = []
zero_shot_errors = 0

for article in tqdm(articles, desc="Zero-shot"):
    try:
        pred = summarizer.summarize(article=article, generation=generation_cfg)
    except Exception as exc:
        pred = ""
        zero_shot_errors += 1
        print(f"Zero-shot generation error: {exc}")
    zero_shot_predictions.append(pred)

print(f"Zero-shot complete: {len(zero_shot_predictions)} predictions, {zero_shot_errors} errors")

Zero-shot:   0%|          | 0/20 [00:00<?, ?it/s]

Zero-shot complete: 20 predictions, 0 errors


In [32]:
# few-shot summarization
few_shot_predictions = []
few_shot_errors = 0

for article in tqdm(articles, desc="Few-shot"):
    try:
        pred = summarizer.summarize(
            article=article,
            generation=generation_cfg,
            few_shot_examples=few_shot_examples,
        )
    except Exception as exc:
        pred = ""
        few_shot_errors += 1
        print(f"Few-shot generation error: {exc}")
    few_shot_predictions.append(pred)

print(f"Few-shot complete: {len(few_shot_predictions)} predictions, {few_shot_errors} errors")

Few-shot:   0%|          | 0/20 [00:00<?, ?it/s]

Few-shot complete: 20 predictions, 0 errors


In [33]:
# compute ROUGE metrics and prepare comparison table
zero_shot_metrics = compute_rouge_batch(references, zero_shot_predictions)
few_shot_metrics = compute_rouge_batch(references, few_shot_predictions)

rows = [
    format_comparison_row(
        model_label="FLAN-T5 Base (Zero-shot)",
        metrics=zero_shot_metrics,
        sample_count=NUM_SAMPLES,
        notes=f"temp={generation_cfg.temperature}, top_k={generation_cfg.top_k}, top_p={generation_cfg.top_p}, errors={zero_shot_errors}",
    ),
    format_comparison_row(
        model_label="FLAN-T5 Base (Few-shot)",
        metrics=few_shot_metrics,
        sample_count=NUM_SAMPLES,
        notes=f"2 demos, temp={generation_cfg.temperature}, errors={few_shot_errors}",
    ),
]

comparison_table_md = to_markdown_table(rows)
comparison_df = pd.DataFrame(rows)
comparison_df

,model,rouge1,rouge2,rougeL,samples,notes
0,FLAN-T5 Base (Zero-shot),0.3265,0.0981,0.2237,20,"temp=0.7, top_k=50, top_p=0.95, errors=0"
1,FLAN-T5 Base (Few-shot),0.0739,0.0000,0.0597,20,"2 demos, temp=0.7, errors=0"


In [34]:
# save baseline results to markdown
run_time = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
content = "\n".join([
    "# Baseline vs Fine-Tuned ROUGE Comparison",
    "",
    "## Baseline (Before Fine-Tuning)",
    "",
    f"Generated at: {run_time}",
    "",
    comparison_table_md,
    "",
    "## Notes",
    "- Dataset: cnn_dailymail test subset (20 samples)",
    "- Base model: google/flan-t5-base",
    "- Fine-tuned section will be appended by notebook 03",
])

COMPARISON_FILE.write_text(content, encoding="utf-8")
print(f"Saved baseline comparison table to: {COMPARISON_FILE}")

Saved baseline comparison table to: d:\GenAI_DeepLearning\domain-summarizer\results\comparison_table.md


In [35]:
# inspect qualitative examples
preview_rows = []
for i in range(min(3, NUM_SAMPLES)):
    preview_rows.append({
        "sample_idx": i,
        "reference": references[i][:300],
        "zero_shot": zero_shot_predictions[i][:300],
        "few_shot": few_shot_predictions[i][:300],
    })

pd.DataFrame(preview_rows)

,sample_idx,reference,zero_shot,few_shot
0,0,CNN's Dr. Sanjay Gupta says we should legalize...,A revolution is taking place everywhere.,Zully Broussard's generosity paired up with bi...
1,1,Child has amassed thousands of Twitter followe...,"Little boy poses for his followers with guns, ...","Zully Broussard, a Californian transplant spec..."
2,2,The presidential hopeful held a town hall meet...,New Jersey Gov Chris Christie is being called ...,"Zully Broussard says, ""I thought I was going t..."


In [36]:
# cleanup
summarizer.unload()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Cleanup complete.")

Cleanup complete.
